# 08 - Model Explainability (SHAP)

## Customer Intelligence Platform

This notebook uses SHAP (SHapley Additive exPlanations) to explain
model predictions at both global and individual levels.

---


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import shap

from src.config import MODELS_DIR, BEST_MODEL_FILE
from src.explain import (
    compute_shap_values,
    plot_shap_summary,
    plot_shap_bar,
    plot_shap_waterfall,
    plot_shap_dependence,
    get_top_drivers,
)

%matplotlib inline


In [ ]:
# Load model and test data
best_model = joblib.load(BEST_MODEL_FILE)
X_test = pd.read_csv(MODELS_DIR / "X_test.csv")
y_test = pd.read_csv(MODELS_DIR / "y_test.csv").squeeze()
print(f"Model: {type(best_model).__name__}")
print(f"Test set: {X_test.shape}")


## 1. Compute SHAP Values


In [ ]:
shap_values, X_sample = compute_shap_values(best_model, X_test, max_samples=500)
print(f"SHAP values computed for {len(X_sample)} samples")


## 2. Global Explanations

### SHAP Summary Plot (Beeswarm)
Each dot represents a customer. Color indicates feature value (red=high, blue=low).
Position on x-axis shows SHAP value (impact on churn prediction).


In [ ]:
fig = plot_shap_summary(shap_values, X_sample, max_display=20)
plt.show()


### SHAP Bar Plot
Mean absolute SHAP values showing overall feature importance.


In [ ]:
fig = plot_shap_bar(shap_values, X_sample, max_display=15)
plt.show()


## 3. SHAP Dependence Plots


In [ ]:
top_features = X_sample.columns[:5].tolist()
for feat in top_features[:3]:
    try:
        fig = plot_shap_dependence(shap_values, X_sample, feat)
        plt.show()
    except Exception as e:
        print(f"Could not plot dependence for {feat}: {e}")


## 4. Local Explanations (Individual Customers)

### High-Risk Customer


In [ ]:
# Find a high-risk customer
y_prob = best_model.predict_proba(X_sample)[:, 1]
high_risk_idx = np.argmax(y_prob)
print(f"Customer {high_risk_idx}: Churn Probability = {y_prob[high_risk_idx]:.2%}")
print(f"\nTop Churn Drivers:")
drivers = get_top_drivers(shap_values, X_sample, high_risk_idx, top_n=5)
print(drivers.to_string(index=False))


In [ ]:
fig = plot_shap_waterfall(shap_values, idx=high_risk_idx)
plt.show()


### Low-Risk Customer


In [ ]:
low_risk_idx = np.argmin(y_prob)
print(f"Customer {low_risk_idx}: Churn Probability = {y_prob[low_risk_idx]:.2%}")
print(f"\nTop Retention Factors:")
drivers = get_top_drivers(shap_values, X_sample, low_risk_idx, top_n=5)
print(drivers.to_string(index=False))


In [ ]:
fig = plot_shap_waterfall(shap_values, idx=low_risk_idx)
plt.show()


## Summary

### Global Insights
- Top churn drivers identified and ranked by SHAP importance
- Feature interactions visible through dependence plots
- Clear distinction between retention factors and churn accelerators

### Local Insights
- Individual predictions can be fully explained
- Each customer gets a personalized "why" for their churn risk
- Enables targeted, evidence-based retention conversations
